# 01-05 - Inside a GeoDataFrame

In 01-02 we called it a table with one extraordinary column and moved on, because a room full of people was waiting for a map. Now we open it up.

We look at rows, columns and the index, we learn the difference between a column and a table, we count categories, we filter with a condition, and we meet missing data. Most of this has nothing to do with geography, which is exactly why it carries over to every dataset you touch this semester.

This notebook is self-paced. Nothing here was covered in class, and nothing here is assessed.

Thirty-five minutes.

In [1]:
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('default')
sns.set_style("whitegrid")

## The table we did not look at

In class we counted the buildings and drew them as a grey smear. Let us actually read them:

In [2]:
buildings = gpd.read_file("data/princeton_buildings.geojson")
buildings.head()

,name,building,geometry
0,NaN,house,"POLYGON ((-74.68918 40.3152, -74.68923 40.3152..."
1,NaN,house,"POLYGON ((-74.68863 40.31549, -74.68857 40.315..."
2,NaN,house,"POLYGON ((-74.68943 40.31573, -74.68951 40.315..."
3,NaN,yes,"POLYGON ((-74.68982 40.31579, -74.68986 40.315..."
4,NaN,house,"POLYGON ((-74.6883 40.31608, -74.68847 40.3158..."


Three columns, `name`, `building` and `geometry`, and an index down the left counting the rows.

Note the `name` column immediately. Several of the first five rows say `None`, meaning there is no value there. Hold that thought; it turns out to be the most interesting thing in this notebook.

## How big, and what kind

Three questions worth asking of any table before you do anything with it:

In [3]:
print(buildings.shape)
print(list(buildings.columns))
print(buildings.dtypes)

(7893, 3)
['name', 'building', 'geometry']
name             str
building         str
geometry    geometry
dtype: object


`(7893, 3)`, i.e. 7893 rows and 3 columns, always in that order.

The dtypes tell you what each column holds. `object` means text, and `geometry` is a type geopandas adds, which is the one that makes this a GeoDataFrame rather than an ordinary table.

Note that `shape`, `columns` and `dtypes` have no brackets. They are properties of the table rather than things it does. Adding brackets to a property is one of the ways to get the `<bound method ...>` output we met in the foundations series.

## A column is not a table

Selecting a column looks simple and has one corner worth knowing:

In [4]:
one_column = buildings["building"]
small_table = buildings[["building"]]

print(type(one_column))
print(type(small_table))

<class 'pandas.Series'>
<class 'pandas.DataFrame'>


A single name in single brackets gives a **Series**, i.e. one column. A list of names in double brackets gives a **DataFrame**, i.e. a table that happens to have one column.

They print differently and they accept different methods, so when something complains that a Series has no attribute you were expecting, this is usually why. The rule: double brackets when you want a table, single when you want a column.

## What is actually in a column

`value_counts` is the fastest way to find out what a column contains, and it should be the first thing you run on any column you have not seen:

In [5]:
buildings["building"].value_counts().head(10)

building
house          4135
yes            2958
university      177
retail          126
garage          107
residential      93
shed             69
dormitory        68
apartments       37
school           28
Name: count, dtype: int64

`house` 4135, `yes` 2958, then `university`, `retail`, `garage` and a long tail.

Stop at `yes`. That is not a kind of building. It is somebody drawing an outline on OpenStreetMap and saying "there is a building here" without saying what sort. Those two categories together are the overwhelming majority:

In [6]:
counts = buildings["building"].value_counts()
share = (counts["house"] + counts["yes"]) / len(buildings) * 100

print(f"{buildings['building'].nunique()} distinct values")
print(f"house and yes together: {share:.1f} percent of every building")

32 distinct values
house and yes together: 89.9 percent of every building


Thirty-two distinct values, and 89.9 percent of the data is in just two of them, one of which means nothing at all.

So a question like "how many shops are there in Princeton" cannot be answered from this column with any confidence. The information is not there for nine buildings in ten. That is not a flaw in our copy of the data; it is what the dataset is.

## Choosing rows

To keep some rows, write a condition and put it inside the table's brackets. The condition produces one True or False per row:

In [7]:
university = buildings[buildings["building"] == "university"]

print(f"{len(university)} university buildings")
university[["name", "building"]].head()

177 university buildings


,name,building
300,Modular Office Space,university
301,Simonyi Hall,university
302,Wolfensohn Hall,university
303,Bloomberg Hall,university
304,Rubenstein Commons,university


177 of them, and now the `name` column has real content, because university buildings are the sort of thing volunteers bother to name.

That contrast is the whole lesson of this notebook, and the next section measures it.

## What is missing

`isna` marks every absent value as True, and summing that counts them:

In [8]:
missing = buildings["name"].isna().sum()
named = buildings["name"].notna().sum()

print(f"named:   {named:,}")
print(f"missing: {missing:,}")
print(f"that is {missing / len(buildings) * 100:.1f} percent with no name at all")

named:   388
missing: 7,505
that is 95.1 percent with no name at all


388 named, 7,505 unnamed, i.e. 95.1 percent of the buildings in Princeton have no name in this dataset.

**A dataset is a record of what somebody bothered to write down, not an inventory of the world.** Princeton does not have 388 buildings worth naming. It has 388 buildings that somebody, at some point, thought worth naming on a volunteer map, and 7505 that are outlines and nothing more.

Note how cheaply we found that out. One method and one line. Running `isna().sum()` on a column you are about to rely on takes five seconds, and it occasionally saves an entire analysis from being quietly meaningless.

## The geometry column

The column that makes this a GeoDataFrame behaves like the others in some ways and not in others.

In [9]:
print(buildings.geom_type.value_counts().to_dict())
print(buildings.crs)
print(buildings.total_bounds)

{'Polygon': 7892, 'MultiPolygon': 1}
EPSG:4326
[-74.7188512  40.3131445 -74.6204158  40.3908891]


7892 Polygons and a single MultiPolygon, i.e. one building that is recorded as several separate pieces, perhaps a structure with a courtyard or two wings counted as one.

Note that a mixed geometry column is normal and occasionally inconvenient. Some operations behave differently on a MultiPolygon, and a dataset that is almost all one type with a handful of exceptions is a classic source of a bug that appears in one row out of eight thousand.

## Measuring the shapes

Because the geometry column knows it holds shapes, it can measure them. But only in a coordinate system where measurement means something, which is why we reproject first, as in 01-03:

In [10]:
in_feet = buildings.to_crs("EPSG:3424")
areas = in_feet.geometry.area

print(f"largest footprint:  {areas.max():,.0f} square feet")
print(f"median footprint:   {areas.median():,.0f} square feet")
print(f"all buildings:      {areas.sum() / 27878400:.2f} square miles")

largest footprint:  206,051 square feet
median footprint:   2,155 square feet
all buildings:      0.91 square miles


The largest single footprint in Princeton is a little over 206,000 square feet, and the median is far smaller, which is what you would expect in a town of mostly houses.

Now let us find out what that largest building is:

In [11]:
largest = buildings.loc[[areas.idxmax()]]
largest[["name", "building"]]

,name,building
1057,NaN,roof


It has no name, and its `building` tag is `roof`.

The largest structure in the dataset is an unnamed roof. That is almost certainly a car park, a covered walkway or something similar, traced by a volunteer who recorded the one thing they were sure of. If you had asked "what is the biggest building in Princeton" and trusted the answer, you would have got something that is arguably not a building at all.

Note that `idxmax` gives the index label of the largest value, and `loc[[...]]` with a list around it returns a table rather than a single row, which is the double-bracket idea again.

## Check your understanding

1. `buildings["building"]` and `buildings[["building"]]` return different things. Which is the table?
2. `buildings["name"].notna().sum()` returns 388. Say in one sentence what that means about Princeton, being careful about what it does not mean.
3. There are 4135 buildings tagged `house` and 2958 tagged `yes`. Which number would you put in a report, and what would you say about the other?
4. Why did we call `to_crs` before `.area`, when we did not need it for `value_counts`?

## Where we are

You can open a GeoDataFrame, see its shape and types, select a column or a sub-table, count categories, filter rows, find missing values, and measure geometry.

The habit to keep is the one that turned this notebook from a tour into a finding: whenever you meet a new column, run `value_counts` and `isna().sum()` on it before you rely on it.

In 01-06 we go back to the thing we have been deferring since 01-03, and explain coordinate reference systems properly.

## Further resources

Nothing in this course requires anything below.

The geopandas user guide covers data structures and the geometry column: https://geopandas.org/en/stable/docs/user_guide/data_structures.html. For the table operations, which are pandas rather than geopandas, the indexing guide is the reference: https://pandas.pydata.org/docs/user_guide/indexing.html.

The buildings are an OpenStreetMap extract, snapshot 2026-08-29, licensed ODbL 1.0, copyright OpenStreetMap contributors: https://www.openstreetmap.org/copyright. The tag vocabulary, including what `yes` means, is documented at https://wiki.openstreetmap.org/wiki/Key:building.